<a href="https://colab.research.google.com/github/AKUN08/tugaskelompokEDA/blob/main/TugasKelompok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---


Nama Anggota
Kelompok

1. Kaifat Dzaky Al Bukhori 2. Muhammad Fauzan Ramadhan


---
Dataset yang Dipilih: dataset_penjualan_kantin


---
Pertanyaan Analisis
Awal

1. Data null 2. data duplikat 3. check datatypes

---

Dugaan Masalah
Kualitas Data: banyak data yang kosong dan datatypenya berbeda beda


---
Rencana Teknik
Pembersihan: fillna untuk penjualan dropna untuk baris yang memiliki nilai null


---
Rencana Manipulasi
Data

Filter: ...... | Sort: ...... | Kolom turunan: ...... |
Groupby/agregasi: ......


---
Jadwal Kerja P3 (Loading & Inspection): Rabu, 9  September 2026 | P4 (Cleaning): Rabu, 9 September 2026 | P5

(Manipulation): ...... | P6 (Uji & Presentasi): ......

---
Pembagian Peran Anggota 1: inspection,cleaning | Anggota 2: loading, manipulation


---







In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('dataset_penjualan_kantin.csv')

print(df.head()) # 5 baris pertama

print(df.info()) # tipe data & non-null

print(df.describe()) # statistik ringkas

print(df.shape) # (baris, kolom)

  id_transaksi     tanggal  nama_produk kategori jumlah_terjual harga_satuan  \
0      TRX0042  2026-08-11   Roti Bakar  Makanan              9         7000   
1      TRX0005  2026-08-03     Gorengan  makanan              2         2000   
2      TRX0011  2026-08-04  Jus Alpukat  Minuman              4          NaN   
3      TRX0035  2026-08-10     Mie Ayam  Makanan            NaN        10000   
4      TRX0007  2026-08-03      Kerupuk    Snack             10      Rp2.000   

  nama_kasir metode_pembayaran  
0   Pak Agus             Tunai  
1     Bu Sri              QRIS  
2    Bu Wati             Tunai  
3   Pak Joko          Transfer  
4    Bu Wati          Transfer  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_transaksi       69 non-null     object
 1   tanggal            69 non-null     object
 2   nama_produk        69 non-

In [ ]:
print(df.isnull().sum())
df['jumlah_terjual'] = df['jumlah_terjual'].fillna(0)
df['harga_satuan'] = df['harga_satuan'].fillna(0)
df = df.dropna(subset=['nama_kasir'])

id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       4
harga_satuan         3
nama_kasir           3
metode_pembayaran    0
dtype: int64


In [ ]:
print(df.duplicated().sum())
df = df.drop_duplicates()

# Handle 'jumlah_terjual' column
df['jumlah_terjual'] = df['jumlah_terjual'].astype(str).str.replace(' pcs', '', regex=False) # Remove ' pcs'
df['jumlah_terjual'] = pd.to_numeric(df['jumlah_terjual'], errors='coerce') # Convert to numeric, coerce errors
df['jumlah_terjual'] = df['jumlah_terjual'].fillna(0).astype(int) # Fill NaNs with 0 and convert to int

df['harga_satuan'] = df['harga_satuan'].astype(str).str.replace('Rp', '', regex=False).str.replace('.', '', regex=False).astype(int)

# Create a dictionary for Indonesian month mapping
indonesian_month_map = {
    'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
    'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
    'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'
}

# Replace Indonesian month names with numerical values in the 'tanggal' column
for indo_month, num_month in indonesian_month_map.items():
    df['tanggal'] = df['tanggal'].astype(str).str.replace(indo_month, num_month, regex=False)

# Standardize date separators (e.g., '14 08 2026' -> '14-08-2026')
df['tanggal'] = df['tanggal'].astype(str).str.replace(' ', '-', regex=False)

# Convert to datetime, allowing mixed formats, dayfirst, and coercing errors
df['tanggal'] = pd.to_datetime(df['tanggal'], format='mixed', dayfirst=True, errors='coerce')

print(df.dtypes)
print(df.shape)

4
id_transaksi                 object
tanggal              datetime64[ns]
nama_produk                  object
kategori                     object
jumlah_terjual                int64
harga_satuan                  int64
nama_kasir                   object
metode_pembayaran            object
dtype: object
(62, 8)


In [ ]:
laris = df[df['jumlah_terjual'] > 20] # filtering
urut = df.sort_values(by='jumlah_terjual', ascending=False) # sorting
df['total_pendapatan'] = df['harga_satuan'] * df['jumlah_terjual'] # kolom turunan
ringkasan = df.groupby('nama_produk')['total_pendapatan'].sum() # agregasi
print(ringkasan)

nama_produk
Bakso                352000
Es Jeruk              36000
Es Teh                93000
Gorengan              18000
Jus Alpukat          440000
Keripik Singkong      72000
Kerupuk               74000
Mie Ayam             510000
Nasi Goreng          840000
Nasi Uduk            168000
Roti Bakar          3633000
Teh Botol            290000
Name: total_pendapatan, dtype: int64
